In [1]:
from google.colab import drive
import sys
drive.mount("/content/drive")
sys.path.append("/content/drive/MyDrive")

Mounted at /content/drive


In [3]:
from rich.traceback import install
install()

import gc
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from skimage import io

from setup_helper import (
    DRIVE_BASE, INPUT_DIR, OUTPUT_DIR,
    download_repos_and_setup,
    show_image, plot_masks,
)
from alignment_helpers import (
    load_image,
    find_roi_with_origins,
    save_search_regions, load_search_regions,
    tile_flm,
    save_tiles, load_tiles_manifest,
    save_images_to_dir,
    filter_with_threshold,
    process_tem_image,
    generate_tem_masks,
    run_lightglue_matching, save_match_records,
    build_alignment_results,
    visualize_overlay,
    save_result_bundle,
    MatchRecord,
)

# ── Inputs ───────────────────────────────────────────────────────────────────
FLM_STACK_PATH = INPUT_DIR / "FLM-stack_JEY002_G3_L3.tif"
TEM_IMAGE_PATH = INPUT_DIR / "JEY002_G3_L3_1950x_t-13.tif"

flm_pixel_nm   = 121.0
tem_pixel_nm   = 6.9
expected_scale = flm_pixel_nm / tem_pixel_nm   # ~17.5

# ── Output tree ──────────────────────────────────────────────────────────────
FLM_FRAMES_DIR = OUTPUT_DIR / "flm_frames"
TEM_DIR        = OUTPUT_DIR / "tem"
MATCHES_DIR    = OUTPUT_DIR / "matches"
RESULTS_DIR    = OUTPUT_DIR / "results"

for d in [FLM_FRAMES_DIR, TEM_DIR, MATCHES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [ ]:
from skimage.filters import threshold_otsu

flm_stack = io.imread(str(FLM_STACK_PATH))

otsu_values = {}
for frame_idx in range(flm_stack.shape[0]):
    green = flm_stack[frame_idx, :, :, 0].astype(float)
    blue  = flm_stack[frame_idx, :, :, 2].astype(float)
    ref  = flm_stack[frame_idx, :, :, 1].astype(float)
    bl_gr = ref
    otsu_values[frame_idx] = threshold_otsu(bl_gr)
    print(f"Frame {frame_idx}: Otsu threshold = {otsu_values[frame_idx]:.2f}")

sharpest_frame = max(otsu_values, key=otsu_values.get)
print(f"\nSharpest frame: {sharpest_frame} (threshold={otsu_values[sharpest_frame]:.2f})")

del flm_stack
gc.collect()

In [ ]:
from skimage.filters import threshold_otsu

flm_stack = io.imread(str(FLM_STACK_PATH))
img_tem   = io.imread(str(TEM_IMAGE_PATH))

otsu_roi = {}
for frame_idx in range(flm_stack.shape[0]):
    flm_frame = flm_stack[frame_idx]
    crops, _ = find_roi_with_origins(flm_frame, img_tem, flm_pixel_nm, tem_pixel_nm, pad_factor=2)

    # compute Otsu on the reflection channel of each ROI crop and average
    values = []
    areas = []
    for crop in crops:
        if crop.max() == crop.min():
            continue
        values.append(threshold_otsu(crop))
        areas.append(crop.shape[0] * crop.shape[1])

    if values:
        otsu_roi[frame_idx] = np.average(values, weights=areas)
    else:
        otsu_roi[frame_idx] = 0

    # otsu_roi[frame_idx] = np.mean(values) if values else 0
    print(f"Frame {frame_idx}: ROI Otsu = {otsu_roi[frame_idx]:.2f}")

sharpest = max(otsu_roi, key=otsu_roi.get)
print(f"\nSharpest frame in ROI: {sharpest}")

del flm_stack, img_tem
gc.collect()

In [5]:
from skimage import exposure, io, measure

def find_roi_with_origins_no_pad(
    img_flm: np.ndarray
) -> tuple[list[np.ndarray], list[tuple[int, int]]]:
    """
    Like find_roi_bl_gr but also returns the (col_offset, row_offset) of each
    crop in the full FLM image — needed for back-projection later.

    Returns
    -------
    crops   : list of 2-D grayscale uint8 crops (reflection channel)
    origins : list of (col_start, row_start) tuples in full-image pixel coords
    """
    green = img_flm[:, :, 0].astype(float)
    blue  = img_flm[:, :, 2].astype(float)
    bl_gr = green + blue

    thresh   = threshold_otsu(bl_gr)
    roi_mask = (bl_gr > thresh).astype(int)

    labelled = measure.label(roi_mask, connectivity=2)
    props    = measure.regionprops(labelled)
    bboxes   = [p.bbox for p in props]

    crops, origins = [], []
    for b in bboxes:
        min_row, min_col, max_row, max_col = b
        r0 = max(0, min_row)
        c0 = max(0, min_col)
        r1 = min(img_flm.shape[0], max_row)
        c1 = min(img_flm.shape[1], max_col)

        # reflection channel = index 1
        crop = img_flm[r0:r1, c0:c1, 1]
        crops.append(crop)
        origins.append((c0, r0))   # (x_offset, y_offset) in full image

    return crops, origins

In [23]:
from skimage.filters import threshold_otsu

flm_stack = io.imread(str(FLM_STACK_PATH))
img_tem   = io.imread(str(TEM_IMAGE_PATH))
"""
There are several observations:
1. just using max or average does not work, these needs to be averaged
2. again using no padding for the crop does not work either
3. all this suggest that laplacian heavily depends on the areas on which it is computed
4. that extra padding which was added seems to server as just the area needed for this to work
5. therefore a better approach seems to collect the overlapping areas and then see which frame for these areas
   has the maximum of laplacian
6. Also those regions should be excluded where the Laplacian does not peak in the middle, but then that raises
   the question if Laplacian is computed on the whole image should I not just select the frames in the middle ?
   or where those frame where the intensity peaks in the middle rather than at the end ?
"""

focus_scores = {}
for frame_idx in range(flm_stack.shape[0]):
    flm_frame = flm_stack[frame_idx]
    # crops, _ = find_roi_with_origins(flm_frame, img_tem, flm_pixel_nm, tem_pixel_nm, pad_factor=2)
    crops, _ = find_roi_with_origins_no_pad(flm_frame)

    values = []
    areas  = []
    for crop in crops:
        if crop.max() == crop.min():
            continue
        crop_u8 = ((crop - crop.min()) / (crop.max() - crop.min()) * 255).astype(np.uint8)
        lap = cv2.Laplacian(crop_u8, cv2.CV_64F).var()
        values.append(lap)
        areas.append(crop.shape[0] * crop.shape[1])

    focus_scores[frame_idx] = np.average(values, weights=areas) if values else 0
    print(f"Frame {frame_idx}: Laplacian var = {focus_scores[frame_idx]:.4f}")

sharpest = max(focus_scores, key=focus_scores.get)
print(f"\nSharpest frame: {sharpest}")

del flm_stack, img_tem
gc.collect()

Frame 0: Laplacian var = 347.9852
Frame 1: Laplacian var = 381.0494
Frame 2: Laplacian var = 389.2945
Frame 3: Laplacian var = 434.7928
Frame 4: Laplacian var = 490.6906
Frame 5: Laplacian var = 519.2060
Frame 6: Laplacian var = 514.6064
Frame 7: Laplacian var = 501.5724
Frame 8: Laplacian var = 486.1194
Frame 9: Laplacian var = 464.0926
Frame 10: Laplacian var = 403.7929
Frame 11: Laplacian var = 351.8904
Frame 12: Laplacian var = 302.7246
Frame 13: Laplacian var = 287.0049
Frame 14: Laplacian var = 261.0921
Frame 15: Laplacian var = 261.7625
Frame 16: Laplacian var = 278.2316
Frame 17: Laplacian var = 252.8072
Frame 18: Laplacian var = 254.0888
Frame 19: Laplacian var = 283.2379
Frame 20: Laplacian var = 293.2418

Sharpest frame: 5


443

### This show that for G3-L8 the following method of trying to find the one in the middle does not work.

What was true in the case of G3-L3 (first Laplacian decreases and then peaks in the middle) does not happen here. The values abruptly increases or decreases until they once again peak in the middle. Given that laplacian and intensity is highly dependent on the areas in which it is calculated it is better to use the full-fledged method.

One method can be to start at the center and then expand outwards until one encounter a larger value on either side which then later is followed by smaller values, however how many small values are needed is not determined by any criteria at all. It is ambiguious at best.

In [31]:
FLM_STACK_PATH = INPUT_DIR / "FLM-stacks-JEY002-G3-L8.tif"

flm_stack = io.imread(str(FLM_STACK_PATH))

lap_vars = []
"""Finds best frame index using Reflection Channel (Index 1)."""
for i in range(flm_stack.shape[0]):
    frame = flm_stack[i]
    # Reflection channel is index 1
    # Check if shape is (C, H, W) or (H, W, C)
    ref_chan = frame[1] if frame.shape[0] == 3 else frame[:, :, 1]

    if ref_chan.max() == ref_chan.min():
        continue

    # Normalize for OpenCV
    ref_u8 = ((ref_chan - ref_chan.min()) / (ref_chan.max() - ref_chan.min()) * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(ref_u8, cv2.CV_64F).var()
    lap_vars.append(lap_var)
    print(f"Frame {i}: Laplacian var = {lap_var:.4f}")

low_idx, high_idx = 0, 0

# find the min index where the low point begins,
# the first few frames then to be unfocused and have the highest laplacian due to having high intensity
for i in range(1, len(lap_vars)):
    if lap_vars[i] > lap_vars[i - 1]:
        low_idx = i - 1
        break

# once the low_idx if found find the peak
for i in range(low_idx + 1, len(lap_vars)):
    if lap_vars[i] < lap_vars[i - 1]:
        high_idx = i - 1
        break

print(low_idx, high_idx)

Frame 0: Laplacian var = 135.4022
Frame 1: Laplacian var = 132.3228
Frame 2: Laplacian var = 124.0603
Frame 3: Laplacian var = 132.7015
Frame 4: Laplacian var = 130.7188
Frame 5: Laplacian var = 129.2841
Frame 6: Laplacian var = 146.3200
Frame 7: Laplacian var = 170.2922
Frame 8: Laplacian var = 184.5177
Frame 9: Laplacian var = 211.7649
Frame 10: Laplacian var = 239.3190
Frame 11: Laplacian var = 251.0133
Frame 12: Laplacian var = 264.1722
Frame 13: Laplacian var = 263.9569
Frame 14: Laplacian var = 268.3795
Frame 15: Laplacian var = 247.4191
Frame 16: Laplacian var = 221.2359
Frame 17: Laplacian var = 188.7401
Frame 18: Laplacian var = 153.3350
Frame 19: Laplacian var = 108.8317
Frame 20: Laplacian var = 85.9194
Frame 21: Laplacian var = 68.5523
Frame 22: Laplacian var = 63.3337
Frame 23: Laplacian var = 59.5018
Frame 24: Laplacian var = 55.5563
Frame 25: Laplacian var = 56.5958
Frame 26: Laplacian var = 54.3143
Frame 27: Laplacian var = 53.2418
Frame 28: Laplacian var = 48.6947
Fram

In [16]:
from shapely.geometry import box

flm_stack = io.imread(str(FLM_STACK_PATH))
img_tem   = io.imread(str(TEM_IMAGE_PATH))

# collect all ROIs across all frames with their bounding boxes and laplacian
all_rois = []  # list of {frame_idx, origin, bbox, laplacian}

for frame_idx in range(flm_stack.shape[0]):
    flm_frame = flm_stack[frame_idx]
    crops, origins = find_roi_with_origins(flm_frame, img_tem, flm_pixel_nm, tem_pixel_nm, pad_factor=1)

    for crop, (ox, oy) in zip(crops, origins):
        if crop.max() == crop.min():
            continue
        h, w = crop.shape[:2]
        crop_u8 = ((crop - crop.min()) / (crop.max() - crop.min()) * 255).astype(np.uint8)
        lap = cv2.Laplacian(crop_u8, cv2.CV_64F).var()
        mean_intensity = crop.mean()
        all_rois.append({
            "frame_idx": frame_idx,
            "origin":    (ox, oy),
            "bbox":      (ox, oy, ox + w, oy + h),  # (x0, y0, x1, y1)
            "laplacian": lap,
            "mean_intensity": mean_intensity,
            "area":      w * h,
        })

del flm_stack, img_tem
gc.collect()

# group by spatial overlap using IoU
def iou(b1, b2):
    ix0 = max(b1[0], b2[0])
    iy0 = max(b1[1], b2[1])
    ix1 = min(b1[2], b2[2])
    iy1 = min(b1[3], b2[3])
    if ix1 <= ix0 or iy1 <= iy0:
        return 0.0
    inter = (ix1 - ix0) * (iy1 - iy0)
    a1    = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2    = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter)

IOU_THRESHOLD = 0.3
groups = []  # list of lists of roi indices
assigned = [False] * len(all_rois)

import networkx as nx

G = nx.Graph()
G.add_nodes_from(range(len(all_rois)))

for i in range(len(all_rois)):
    for j in range(i + 1, len(all_rois)):
        if iou(all_rois[i]["bbox"], all_rois[j]["bbox"]) >= IOU_THRESHOLD:
            G.add_edge(i, j)

groups = list(nx.connected_components(G))

best_per_group = []
for group in groups:
    best_idx = max(group, key=lambda i: all_rois[i]["laplacian"] * all_rois[i]["mean_intensity"])
    best_roi = all_rois[best_idx]
    best_per_group.append(best_roi)
    print(f"ROI group size={len(group):2d}  "
          f"best frame={best_roi['frame_idx']:2d}  "
          f"laplacian={best_roi['laplacian']:.2f}  "
          f"mean_intensity={best_roi['mean_intensity']:.2f}  "
          f"bbox={best_roi['bbox']}")

frames_to_process = sorted(set(r["frame_idx"] for r in best_per_group))
print(f"\nFrames to process: {frames_to_process}")

ROI group size=21  best frame= 6  laplacian=206.66  mean_intensity=730.07  bbox=(1506, 0, 2000, 351)
ROI group size=21  best frame=11  laplacian=119.05  mean_intensity=699.47  bbox=(825, 365, 1770, 1599)
ROI group size=21  best frame= 0  laplacian=314.85  mean_intensity=552.77  bbox=(1759, 1799, 2000, 2000)
ROI group size=19  best frame=10  laplacian=240.28  mean_intensity=501.86  bbox=(567, 1741, 897, 2000)

Frames to process: [0, 6, 10, 11]


In [24]:
# check all frames for the top right corner ROI
corner_rois = [roi for roi in all_rois if iou(roi["bbox"], (1759, 1799, 2000, 2000)) >= IOU_THRESHOLD]
corner_rois_sorted = sorted(corner_rois, key=lambda r: r["frame_idx"])

for roi in corner_rois_sorted:
    print(f"Frame {roi['frame_idx']:2d}: laplacian={roi['laplacian']:.2f}  mean_intensity={roi['mean_intensity']:.2f}  bbox={roi['bbox']}")

Frame  0: laplacian=107.28  mean_intensity=706.52  bbox=(820, 365, 1737, 1596)
Frame  1: laplacian=101.29  mean_intensity=705.95  bbox=(822, 365, 1737, 1597)
Frame  2: laplacian=92.31  mean_intensity=705.27  bbox=(822, 364, 1737, 1598)
Frame  3: laplacian=95.10  mean_intensity=703.88  bbox=(822, 364, 1747, 1598)
Frame  4: laplacian=92.99  mean_intensity=703.49  bbox=(823, 364, 1748, 1598)
Frame  5: laplacian=93.54  mean_intensity=702.67  bbox=(823, 364, 1749, 1598)
Frame  6: laplacian=94.62  mean_intensity=702.45  bbox=(823, 365, 1750, 1598)
Frame  7: laplacian=98.84  mean_intensity=703.15  bbox=(824, 365, 1750, 1598)
Frame  8: laplacian=105.90  mean_intensity=700.20  bbox=(824, 365, 1769, 1598)
Frame  9: laplacian=111.33  mean_intensity=699.52  bbox=(825, 365, 1769, 1598)
Frame 10: laplacian=114.59  mean_intensity=699.03  bbox=(824, 365, 1770, 1599)
Frame 11: laplacian=119.05  mean_intensity=699.47  bbox=(825, 365, 1770, 1599)
Frame 12: laplacian=84.78  mean_intensity=629.37  bbox=(0,

In [18]:
def has_interior_peak(group_rois):
    """Returns True if the Laplacian peaks at an interior frame, not at the edges."""
    sorted_by_frame = sorted(group_rois, key=lambda r: r["frame_idx"])
    values = [r["laplacian"] for r in sorted_by_frame]
    peak_idx = np.argmax(values)
    # peak must not be at the first or last frame
    return 0 < peak_idx < len(values) - 1

# filter groups
valid_best_per_group = []
for group, best_roi in zip(groups, best_per_group):
    group_rois = [all_rois[i] for i in group]
    if has_interior_peak(group_rois):
        valid_best_per_group.append(best_roi)
        print(f"Valid ROI: best frame={best_roi['frame_idx']}  laplacian={best_roi['laplacian']:.2f}  bbox={best_roi['bbox']}")
    else:
        print(f"Rejected ROI (no interior peak): bbox={best_roi['bbox']}")

frames_to_process = sorted(set(r["frame_idx"] for r in valid_best_per_group))
print(f"\nFrames to process: {frames_to_process}")

Valid ROI: best frame=6  laplacian=206.66  bbox=(1506, 0, 2000, 351)
Valid ROI: best frame=11  laplacian=119.05  bbox=(825, 365, 1770, 1599)
Rejected ROI (no interior peak): bbox=(1759, 1799, 2000, 2000)
Valid ROI: best frame=10  laplacian=240.28  bbox=(567, 1741, 897, 2000)

Frames to process: [6, 10, 11]


In [ ]:
# input from the experimenter — approximate FLM pixel location of TEM field of view
tem_location_in_flm = (1200, 800)  # (x, y) — experimenter fills this in

# find which ROI group contains this point
def point_in_bbox(point, bbox):
    x, y = point
    x0, y0, x1, y1 = bbox
    return x0 <= x <= x1 and y0 <= y <= y1

relevant_group = None
for group, best_roi in zip(groups, best_per_group):
    if point_in_bbox(tem_location_in_flm, best_roi["bbox"]):
        relevant_group = group
        relevant_roi   = best_roi
        break

if relevant_group is None:
    print("Point not found in any ROI — check tem_location_in_flm")
else:
    best_frame = relevant_roi["frame_idx"]
    print(f"Relevant ROI: {relevant_roi['bbox']}")
    print(f"Best frame: {best_frame}  laplacian={relevant_roi['laplacian']:.2f}")
    frames_to_process = [best_frame]